# 🌸 Nayari Dataset Builder  v3
**Run locally.** Scans `dataset/` (including subdirectories) for `.md`, `.txt`, `.pdf`, and `.json` files, applies a robust multi-format parser, deduplicates, quality-filters, and exports `nayari_dataset.json`.  Then auto-uploads to Kaggle.

| Feature | Details |
|---|---|
| Speaker formats | `Name:`, `**Name**:`, `> Name:`, `[Name]:` |
| Scene splitting | Blank-line gaps, `---END---`, `<end>`, `===` dividers |
| Auto-aliases | Unknown speakers scanned from filenames & content |
| Quality filter | Min 2 turns, min 10 chars/message |
| Deduplication | MD5 fingerprint on normalised content |
| OOC stripping | `[OOC: …]` and `(OOC …)` blocks removed |
| Source tagging | Every conversation records its origin file |
| Token estimate | Rough GPT-style token count in the stats cell |


## 📦 Step 0 — Install & Import

In [1]:
%pip install pdfplumber kaggle -q

import re, json, os, shutil, subprocess, sys, hashlib, warnings
import pdfplumber
from pathlib import Path
from collections import defaultdict

warnings.filterwarnings("ignore", message="Could not get FontBBox")

DATASET_DIR = Path("dataset")
OUTPUT_FILE = Path("nayari_dataset.json")

# ── pretty directory scan ────────────────────────────────────────────────────
all_files = sorted(
    f for f in DATASET_DIR.rglob("*")
    if f.is_file() and f.suffix.lower() in {".md", ".txt", ".pdf", ".json"}
)
print(f"Found {len(all_files)} source file(s) in dataset/:\n")
for f in all_files:
    rel = f.relative_to(DATASET_DIR)
    print(f"  [{f.suffix.upper():5}] {rel}  ({f.stat().st_size/1024:.1f} KB)")


Note: you may need to restart the kernel to use updated packages.
Found 196 source file(s) in dataset/:

  [.MD  ] set 1\md files\chats\Nayari_chat_1.md  (4.8 KB)
  [.MD  ] set 1\md files\chats\Nayari_chat_10.md  (4.8 KB)
  [.MD  ] set 1\md files\chats\Nayari_chat_11.md  (4.7 KB)
  [.MD  ] set 1\md files\chats\Nayari_chat_12.md  (4.6 KB)
  [.MD  ] set 1\md files\chats\Nayari_chat_13.md  (4.5 KB)
  [.MD  ] set 1\md files\chats\Nayari_chat_14.md  (4.7 KB)
  [.MD  ] set 1\md files\chats\Nayari_chat_15.md  (1.2 KB)
  [.MD  ] set 1\md files\chats\Nayari_chat_16.md  (1.1 KB)
  [.MD  ] set 1\md files\chats\Nayari_chat_17.md  (1.2 KB)
  [.MD  ] set 1\md files\chats\Nayari_chat_18.md  (1.2 KB)
  [.MD  ] set 1\md files\chats\Nayari_chat_19.md  (1.4 KB)
  [.MD  ] set 1\md files\chats\Nayari_chat_2.md  (4.2 KB)
  [.MD  ] set 1\md files\chats\Nayari_chat_20.md  (1.0 KB)
  [.MD  ] set 1\md files\chats\Nayari_chat_21.md  (1.2 KB)
  [.MD  ] set 1\md files\chats\Nayari_chat_22.md  (1.2 KB)
  [.MD  ] se

## 1 — Character Details

In [2]:
# Locate the details file anywhere inside dataset/
details_candidates = list(DATASET_DIR.rglob("*details*"))
if not details_candidates:
    print("⚠️  No details file found — character metadata will be empty.")
    char_name = char_type = char_gender = char_traits = char_personality = ""
else:
    details_text = details_candidates[0].read_text(encoding="utf-8", errors="replace")

    def extract_field(text, field):
        m = re.search(rf"\*\*{field}\*\*:?\s*(.+)", text)
        return m.group(1).strip() if m else ""

    char_name        = extract_field(details_text, "Name")
    char_type        = extract_field(details_text, "Type")
    char_gender      = extract_field(details_text, "Gender")
    char_traits      = extract_field(details_text, "Traits")
    char_personality = extract_field(details_text, "Personally")
    print(f"Character : {char_name} | {char_type} | {char_gender}")
    print(f"Details file: {details_candidates[0].relative_to(DATASET_DIR)}")

lore_sections = []


Character : Nayari | Kemonomimi | Female (Cis)
Details file: set 1\md files\lore\Nayari_Details.md


## 2 — Robust Multi-Format Parser

Handles every speaker format seen in the repo:

```
Name: text           ← plain
**Name**: text       ← bold markdown
> Name: text         ← blockquote
[Name]: text         ← bracket
```

Scene splitting triggers on: `--- END ---`, `=== break ===`, `<end>`, or **3+ blank lines**.
OOC annotations `[OOC: …]` and `(OOC …)` are stripped from message content.

In [3]:
# ── Core aliases (extended automatically per-file below) ────────────────────
BASE_USER_ALIASES      = {"me", "you", "tiaya"}
BASE_ASSISTANT_ALIASES = {"nayari", "nayri", "aura"}

# Universal speaker line pattern — covers all 4 formats
SPEAKER_RE = re.compile(
    r"""^
    (?:\*{1,2}|\[|>\s*)?          # optional: ** or [ or >
    ([A-Za-z][A-Za-z0-9 _\'\-]*)   # speaker name
    (?:\*{1,2}|\])?                 # optional closing ** or ]
    :\s*(.*)                         # colon + rest of line
    $""",
    re.VERBOSE,
)

END_RE = re.compile(
    r"^[-=*]{3,}\s*(<?(end|END|break|BREAK|scene|SCENE)>?)?\s*[-=*]{0,}$"
)

OOC_RE = re.compile(r"\[OOC:?[^\]]*\]|\(OOC[^)]*\)", re.IGNORECASE)


def _strip_ooc(text: str) -> str:
    return OOC_RE.sub("", text).strip()


def _inject_scene_ends(text: str) -> str:
    """Replace 3+ consecutive blank lines with a synthetic END marker."""
    return re.sub(r"(\r?\n){3,}", "\n--- END ---\n", text)


def _detect_aliases(text: str, filename: str):
    """
    Heuristically discover user/assistant aliases from the file.
    Any speaker that appears >= 2 times AND whose name contains a
    known assistant keyword is mapped to assistant; everything else
    that appears frequently becomes a user alias candidate.
    """
    counts = defaultdict(int)
    for line in text.splitlines():
        m = SPEAKER_RE.match(line.strip())
        if m:
            counts[m.group(1).strip().lower()] += 1

    user_extra, asst_extra = set(), set()
    asst_keywords = {"nayari", "nayri", "aura", "goddess"}
    for name, cnt in counts.items():
        if cnt < 2:
            continue
        if any(kw in name for kw in asst_keywords):
            asst_extra.add(name)
        else:
            user_extra.add(name)
    return user_extra, asst_extra


def parse_chat_text(text: str, filename: str = ""):
    """
    Full-featured chat parser.
    Returns list of {messages, source, turns} dicts.
    """
    user_aliases = BASE_USER_ALIASES.copy()
    asst_aliases = BASE_ASSISTANT_ALIASES.copy()
    u_extra, a_extra = _detect_aliases(text, filename)
    user_aliases |= u_extra
    asst_aliases |= a_extra

    text = _inject_scene_ends(text)
    lines = text.splitlines()
    conversations, current_messages = [], []
    current_role, buf = None, []
    skipped_lines = []

    def flush():
        nonlocal buf
        if current_role and buf:
            raw = " ".join(" ".join(buf).split()).strip()
            content = _strip_ooc(raw)
            if len(content) >= 10:          # quality: min 10 chars
                current_messages.append({"role": current_role, "content": content})
            elif content:
                skipped_lines.append(f"  ⚠ Short message ({len(content)} chars): {content!r}")
        buf = []

    def save():
        if len(current_messages) >= 2:      # quality: min 2 turns
            conversations.append({
                "messages": list(current_messages),
                "source": filename,
            })
        current_messages.clear()

    for line in lines:
        s = line.strip()
        if not s:
            continue
        # heading / metadata lines — skip
        if s.startswith(("##", "#", "---", "===")) and not SPEAKER_RE.match(s):
            if END_RE.match(s) or "<end>" in s.lower():
                flush(); save(); current_role = None
            continue
        if END_RE.match(s) or "<end>" in s.lower() or "<--- end" in s.lower():
            flush(); save(); current_role = None; continue

        m = SPEAKER_RE.match(s)
        if m:
            sp   = m.group(1).strip().lower()
            rest = m.group(2).strip()
            if sp in user_aliases:
                flush(); current_role = "user"; buf = [rest] if rest else []
            elif sp in asst_aliases:
                flush(); current_role = "assistant"; buf = [rest] if rest else []
            else:
                # unknown speaker — carry on accumulating if mid-turn
                if current_role:
                    buf.append(s)
        else:
            if current_role:
                buf.append(s)

    flush(); save()

    if skipped_lines:
        print(f"    [{filename}] quality skips:")
        for w in skipped_lines[:5]:
            print(w)

    return conversations


# ── deduplication ─────────────────────────────────────────────────────────────
def _fingerprint(conv: dict) -> str:
    norm = " ".join(
        m["content"][:80].lower()
        for m in conv["messages"]
    )
    return hashlib.md5(norm.encode()).hexdigest()


def deduplicate(convs: list) -> list:
    seen, out = set(), []
    for c in convs:
        fp = _fingerprint(c)
        if fp not in seen:
            seen.add(fp)
            out.append(c)
    return out


# ── PDF extractor ─────────────────────────────────────────────────────────────
def extract_pdf(path: Path):
    chat_convs, lore_chunks = [], []
    try:
        with pdfplumber.open(path) as pdf:
            print(f"  {path.name}: {len(pdf.pages)} page(s)")
            for i, page in enumerate(pdf.pages):
                text = page.extract_text() or ""
                if not text.strip():
                    print(f"    Page {i+1}: empty — skipped"); continue
                convs = parse_chat_text(text, path.name)
                if convs:
                    print(f"    Page {i+1}: CHAT  — {len(convs)} scene(s)")
                    chat_convs.extend(convs)
                else:
                    cleaned = re.sub(r"\n{3,}", "\n\n", text).strip()
                    lore_chunks.append(cleaned)
                    print(f"    Page {i+1}: LORE  — {len(cleaned)} chars")
    except Exception as exc:
        print(f"  ❌ Failed to read {path.name}: {exc}")
    return chat_convs, lore_chunks


def is_details_file(path: Path) -> bool:
    return "details" in path.name.lower() or "detail" in path.name.lower()


print("✅ Parser v3 ready — multi-format, auto-alias, OOC-strip, dedup.")


✅ Parser v3 ready — multi-format, auto-alias, OOC-strip, dedup.


## 3 — Parse All Source Files (.md / .txt / .pdf / .json)

In [6]:
import json
from pathlib import Path

# Data storage
processed_data = {
    "conversations": [],
    "lore": [],
    "failed": []
}

def identify_content_type(text, file_path):
    """
    Heuristic: Checks if text looks like a chat (Name: Message) 
    or Lore (long descriptive paragraphs).
    """
    lines = [l.strip() for l in text.split('\n') if len(l.strip()) > 2]
    if not lines: return "unknown"
    
    # Check for chat patterns: "Name:", "[Name]", "<Name>"
    chat_indicators = 0
    sample_lines = lines[:20]
    for line in sample_lines:
        if ":" in line[:20] or (line.startswith("[") and "]" in line[:20]):
            chat_indicators += 1
            
    # If more than 30% of lines look like dialogue, it's likely a chat
    return "chat" if (chat_indicators / len(sample_lines)) > 0.3 else "lore"

print(f"🧠 Intelligence Engine: Processing {len(all_files)} files...\n")

for path in all_files:
    rel = str(path.relative_to(DATASET_DIR))
    ext = path.suffix.lower()
    
    try:
        # --- 1. JSON PROCESSING (Smart Key Detection) ---
        if ext == ".json":
            data = json.loads(path.read_text(encoding="utf-8", errors="replace"))
            
            # Is it a Worldbook/Lorebook? (Common in AI tools like SillyTavern)
            if any(k in str(data).lower() for k in ["entries", "worldbook", "character_book"]):
                processed_data["lore"].append({"source": rel, "content": data, "type": "json_lore"})
                print(f"  [JSON-LORE] {rel}")
            
            # Is it a standard chat log?
            else:
                convs = []
                if isinstance(data, list): convs = [c for c in data if "messages" in c]
                elif isinstance(data, dict):
                    if "conversations" in data: convs = data["conversations"]
                    elif "messages" in data: convs = [data]
                
                for c in convs: c["source"] = rel
                processed_data["conversations"].extend(convs)
                print(f"  [JSON-CHAT] {rel}: {len(convs)} convs")

        # --- 2. PDF PROCESSING (Content-Based Routing) ---
        elif ext == ".pdf":
            chat_convs, lore_chunks = extract_pdf(path)
            
            # Smart Routing: If it has many chat convs, it's a chat file
            if len(chat_convs) > len(lore_chunks):
                processed_data["conversations"].extend(chat_convs)
                print(f"  [PDF-CHAT ] {rel}: {len(chat_convs)} dialogues")
            else:
                name = path.stem
                text = "\n\n".join(lore_chunks)
                processed_data["lore"].append({"source": rel, "name": name, "text": text})
                print(f"  [PDF-LORE ] {rel}: {len(text)} chars")

        # --- 3. TEXT/MD PROCESSING (Heuristic Analysis) ---
        elif ext in {".md", ".txt"}:
            raw_text = path.read_text(encoding="utf-8", errors="replace")
            content_type = identify_content_type(raw_text, path)
            
            if content_type == "chat":
                convs = parse_chat_text(raw_text, rel)
                processed_data["conversations"].extend(convs)
                print(f"  [TXT-CHAT ] {rel}: {len(convs)} convs")
            else:
                processed_data["lore"].append({"source": rel, "text": raw_text})
                print(f"  [TXT-LORE ] {rel}: Descriptive content detected")

    except Exception as exc:
        print(f"  ❌  ERROR {rel}: {exc}")
        processed_data["failed"].append(rel)

# Final Smart Summary
print("\n" + "="*30)
print("FINAL DATASET BREAKDOWN")
print("="*30)
print(f"Total Conversations: {len(processed_data['conversations'])}")
print(f"Total Lore Entries:  {len(processed_data['lore'])}")
if processed_data["failed"]:
    print(f"Failed Files:       {len(processed_data['failed'])}")

🧠 Intelligence Engine: Processing 192 files...

  [TXT-CHAT ] set 1\md files\chats\Nayari_chat_1.md: 1 convs
    [set 1\md files\chats\Nayari_chat_10.md] quality skips:
  ⚠ Short message (5 chars): 'Rain?'
  [TXT-CHAT ] set 1\md files\chats\Nayari_chat_10.md: 4 convs
  [TXT-CHAT ] set 1\md files\chats\Nayari_chat_11.md: 4 convs
    [set 1\md files\chats\Nayari_chat_12.md] quality skips:
  ⚠ Short message (9 chars): 'Ballpark.'
  ⚠ Short message (5 chars): 'Yeah?'
  [TXT-CHAT ] set 1\md files\chats\Nayari_chat_12.md: 4 convs
    [set 1\md files\chats\Nayari_chat_13.md] quality skips:
  ⚠ Short message (7 chars): 'And...?'
  [TXT-CHAT ] set 1\md files\chats\Nayari_chat_13.md: 4 convs
  [TXT-CHAT ] set 1\md files\chats\Nayari_chat_14.md: 4 convs
  [TXT-CHAT ] set 1\md files\chats\Nayari_chat_15.md: 1 convs
  [TXT-CHAT ] set 1\md files\chats\Nayari_chat_16.md: 1 convs
  [TXT-CHAT ] set 1\md files\chats\Nayari_chat_17.md: 1 convs
  [TXT-CHAT ] set 1\md files\chats\Nayari_chat_18.md: 1 convs

Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats


    Page 2: LORE  — 1568 chars
  [PDF-LORE ] set 1\pdf files\herfiles\Discovery .pdf: 4163 chars
  Her Beliefs .pdf: 2 page(s)
    Page 1: LORE  — 1408 chars
    Page 2: LORE  — 402 chars
  [PDF-LORE ] set 1\pdf files\herfiles\Her Beliefs .pdf: 1812 chars

FINAL DATASET BREAKDOWN
Total Conversations: 99
Total Lore Entries:  98


## 4 — Lore → Training Conversations

Each PDF lore section is chunked into paragraphs.
Each chunk is paired with a varied natural-language prompt so the model learns
to recall Nayari's lore organically. Multiple prompts per chunk for augmentation.

In [13]:
import random
import re

# Categories of prompts to make the model more versatile
PROMPT_TEMPLATES = {
    "identity": [
        "Who are you?", "Tell me about yourself.", "Describe your background.",
        "What is your story?", "How would you define your identity?"
    ],
    "knowledge": [
        "Tell me about {subject}.", "What can you share regarding {subject}?",
        "Explain the significance of {subject}.", "Provide some context on {subject}.",
        "What do the records say about {subject}?"
    ],
    "general": [
        "Tell me more.", "Share an entry from the lore.", 
        "What else is there to know?", "Continue the history."
    ]
}

def clean_pdf_text(text):
    """Cleans common PDF extraction artifacts."""
    # Fix hyphenated words at line breaks (e.g., "en- \n vironment" -> "environment")
    text = re.sub(r'(\w+)-\s*\n\s*(\w+)', r'\1\2', text)
    # Replace multiple newlines with a standard double newline
    text = re.sub(r'\n\s*\n', '\n\n', text)
    # Replace single newlines with spaces (common in PDF columns)
    text = re.sub(r'(?<!\n)\n(?!\n)', ' ', text)
    return text.strip()

def extract_subject(text):
    """Heuristic to find the 'Subject' of a chunk for better prompting."""
    # Try to grab the first capitalized phrase or the first sentence's noun
    match = re.search(r'([A-Z][a-z]+(?:\s[A-Z][a-z]+)*)', text)
    if match:
        return match.group(1)
    return None

def lore_to_convs_smart(sections, min_chars=200, max_chars=1000, augment=False):
    convs = []
    
    for section in sections:
        # 1. Clean the text
        raw_text = clean_pdf_text(section["text"])
        source = section.get("source", "Unknown Source")
        
        # 2. Smart Chunking (Recursive-ish approach)
        paragraphs = [p.strip() for p in raw_text.split("\n\n") if p.strip()]
        chunks = []
        current_chunk = ""
        
        for para in paragraphs:
            if len(current_chunk) + len(para) < max_chars:
                current_chunk += ("\n\n" + para if current_chunk else para)
            else:
                if len(current_chunk) >= min_chars:
                    chunks.append(current_chunk)
                current_chunk = para
        if current_chunk:
            chunks.append(current_chunk)

        # 3. Generate Conversations
        for chunk in chunks:
            # Determine subject for dynamic prompting
            subject = extract_subject(chunk)
            
            # Select prompt type
            if "I am" in chunk[:50] or "My name" in chunk[:50]:
                templates = PROMPT_TEMPLATES["identity"]
            elif subject and len(subject) > 3:
                templates = [t.format(subject=subject) for t in PROMPT_TEMPLATES["knowledge"]]
            else:
                templates = PROMPT_TEMPLATES["general"]

            # Augmentation logic
            num_variations = 2 if augment else 1
            selected_prompts = random.sample(templates, min(num_variations, len(templates)))

            for prompt in selected_prompts:
                convs.append({
                    "messages": [
                        {"role": "user", "content": prompt},
                        {"role": "assistant", "content": chunk},
                    ],
                    "metadata": {
                        "source": source,
                        "subject": subject,
                        "length": len(chunk)
                    }
                })
                
    return convs

# Example Usage:
# lore_sections = [{"text": "The Kingdom of Eldoria was founded in... ", "source": "History_Book.pdf"}]
# smart_convs = lore_to_convs_smart(lore_sections, augment=True)

## 5 — Deduplication & Quality Filter

In [16]:
all_raw = processed_data['conversations']
md_conversations = [c for c in all_raw if c.get('source', '').lower().endswith('.md')]
txt_conversations = [c for c in all_raw if c.get('source', '').lower().endswith('.txt')]
pdf_conversations = [c for c in all_raw if c.get('source', '').lower().endswith('.pdf')]
json_conversations = [c for c in all_raw if c.get('source', '').lower().endswith('.json')]

all_conversations = all_raw + lore_conversations

before = len(all_conversations)
all_conversations = deduplicate(all_conversations)
after  = len(all_conversations)

print(f"Before dedup : {before}")
print(f"After  dedup : {after}  ({before - after} duplicates removed)")

# Sanity-check: flag any conversation with a very long single message
WARN_CHARS = 3000
flagged = [
    (i+1, m["role"], len(m["content"]))
    for i, c in enumerate(all_conversations)
    for m in c["messages"]
    if len(m["content"]) > WARN_CHARS
]
if flagged:
    print(f"\n⚠ {len(flagged)} message(s) exceed {WARN_CHARS} chars (may be lore, review if unexpected):")
    for idx, role, length in flagged[:8]:
        print(f"  Conv {idx} [{role}]: {length} chars")
else:
    print("\n✅ No oversized messages.")


Before dedup : 99
After  dedup : 99  (0 duplicates removed)

✅ No oversized messages.


## 6 — Preview & Statistics Dashboard

In [17]:
all_raw = processed_data['conversations']
md_conversations = [c for c in all_raw if c.get('source', '').lower().endswith('.md')]
txt_conversations = [c for c in all_raw if c.get('source', '').lower().endswith('.txt')]
pdf_conversations = [c for c in all_raw if c.get('source', '').lower().endswith('.pdf')]
json_conversations = [c for c in all_raw if c.get('source', '').lower().endswith('.json')]

all_conversations = all_raw + lore_conversations

before = len(all_conversations)
all_conversations = deduplicate(all_conversations)
after  = len(all_conversations)

print(f"Before dedup : {before}")
print(f"After  dedup : {after}  ({before - after} duplicates removed)")

# Sanity-check: flag any conversation with a very long single message
WARN_CHARS = 3000
flagged = [
    (i+1, m["role"], len(m["content"]))
    for i, c in enumerate(all_conversations)
    for m in c["messages"]
    if len(m["content"]) > WARN_CHARS
]
if flagged:
    print(f"\n⚠ {len(flagged)} message(s) exceed {WARN_CHARS} chars (may be lore, review if unexpected):")
    for idx, role, length in flagged[:8]:
        print(f"  Conv {idx} [{role}]: {length} chars")
else:
    print("\n✅ No oversized messages.")


Before dedup : 99
After  dedup : 99  (0 duplicates removed)

✅ No oversized messages.


## 7 — Export JSON

In [15]:
from datetime import datetime, timezone

dataset_json = {
    "meta": {
        "version": 3,
        "built_at": datetime.now(timezone.utc).isoformat(),
        "total_conversations": len(all_conversations),
        "sources": {
            "markdown": len(md_conversations),
            "txt":      len(txt_conversations),
            "pdf":      len(pdf_conversations),
            "json":     len(json_conversations),
            "lore":     len(lore_conversations),
        },
    },
    "character": {
        "name": char_name, "type": char_type, "gender": char_gender,
        "traits": char_traits, "personality": char_personality,
        "lore_sections": lore_sections,
    },
    "conversations": all_conversations,
}

OUTPUT_FILE.write_text(
    json.dumps(dataset_json, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
size_kb = OUTPUT_FILE.stat().st_size / 1024
print(f"✅ Saved → {OUTPUT_FILE.resolve()}")
print(f"   {len(all_conversations)} conversations | {size_kb:.1f} KB")


✅ Saved → T:\Documents\Github Desktop\Nayari-AI\nayari_dataset.json
   99 conversations | 222.4 KB


## 8 — Upload to Kaggle via API

**Before running:**
1. Go to [kaggle.com](https://kaggle.com) → Profile → **Settings** → **API** → **Create New Token**
2. A `kaggle.json` downloads — open it and copy the `username` and `key` values
3. Paste them into the two variables below

> The dataset is created as **private** by default.


In [ ]:
# ── FILL THESE IN ──────────────────────────────────────────────────────────
KAGGLE_USERNAME = "kaggle_username"   # from kaggle.json
KAGGLE_KEY      = "kaggle_key"        # from kaggle.json
DATASET_TITLE   = "nayari-dataset"   # slug used in the Kaggle URL
IS_PUBLIC       = False               # True to make the dataset public
# ───────────────────────────────────────────────────────────────────────────

kaggle_dir = Path.home() / ".kaggle"
kaggle_dir.mkdir(exist_ok=True)
creds_file = kaggle_dir / "kaggle.json"
creds_file.write_text(
    json.dumps({"username": KAGGLE_USERNAME, "key": KAGGLE_KEY}),
    encoding="utf-8",
)
creds_file.chmod(0o600)
print("✅ Credentials written.")

STAGING = Path("_kaggle_upload")
shutil.rmtree(STAGING, ignore_errors=True)
STAGING.mkdir()
shutil.copy(OUTPUT_FILE, STAGING / OUTPUT_FILE.name)

(STAGING / "dataset-metadata.json").write_text(
    json.dumps({
        "title":     DATASET_TITLE,
        "id":        f"{KAGGLE_USERNAME}/{DATASET_TITLE}",
        "licenses":  [{"name": "CC0-1.0"}],
        "isPrivate": not IS_PUBLIC,
    }, indent=2),
    encoding="utf-8",
)
print(f"✅ Staging folder ready: {STAGING.resolve()}")

def run_kaggle(*args):
    return subprocess.run(["kaggle", *args], capture_output=True, text=True)

result  = run_kaggle("datasets", "create", "-p", str(STAGING), "--dir-mode", "zip")
combined = (result.stdout + result.stderr).lower()

if "already exists" in combined or "403" in combined:
    print("Dataset already exists — pushing a new version...")
    result = run_kaggle(
        "datasets", "version", "-p", str(STAGING),
        "-m", f"v3 build — {len(all_conversations)} conversations",
        "--dir-mode", "zip",
    )

print(result.stdout or result.stderr)

if result.returncode == 0:
    vis = "public" if IS_PUBLIC else "private"
    url = f"https://www.kaggle.com/datasets/{KAGGLE_USERNAME}/{DATASET_TITLE}"
    print(f"🎉 Upload successful! ({vis})")
    print(f"   Dataset URL: {url}")
    print()
    print("📋 Next steps in your Kaggle training notebook:")
    print("   1. Open nayari_train.ipynb on Kaggle")
    print(f"   2. Click '+ Add Data' → search '{DATASET_TITLE}' → Add")
    print("   3. Set Accelerator = GPU T4 x2, Internet = On → Run All")
else:
    print("❌ Upload failed. Double-check KAGGLE_USERNAME and KAGGLE_KEY.")

shutil.rmtree(STAGING, ignore_errors=True)
